In [1]:
import pandas as pd 
import numpy as np

In [35]:
# laoding the pre processed data for feature extraction
df = pd.read_csv('C:\\Users\\Aman\\Desktop\\kifyaw4\\data\\processed\\train_processed.csv')
df.columns

Index(['Date', 'Store', 'DayOfWeek', 'Sales', 'Customers', 'Open', 'Promo',
       'SchoolHoliday', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b',
       'StateHoliday_c'],
      dtype='object')

In [3]:
df.head()

,Date,Store,DayOfWeek,Sales,Customers,Open,Promo,SchoolHoliday,StateHoliday_0,StateHoliday_a,StateHoliday_b,StateHoliday_c
0,2015-07-31,1.0,5.0,5263.0,555.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
1,2015-07-31,2.0,5.0,6064.0,625.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
2,2015-07-31,3.0,5.0,8314.0,821.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
3,2015-07-31,4.0,5.0,13995.0,1498.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
4,2015-07-31,5.0,5.0,4822.0,559.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0


In [40]:
df['Date'] = pd.to_datetime(df['Date'])

In [5]:
# df_sampled = df.sample(100, random_state=42)

In [30]:
holiday_a = df[df['StateHoliday_a']== 1]['Date'].unique()
holiday_b = df[df['StateHoliday_b']== 1]['Date'].unique()
holiday_c = df[df['StateHoliday_c']== 1]['Date'].unique()

1. Weekdays and Weekends
Determine whether the date falls on a weekday or weekend.

In [41]:
df['IsWeekday'] = df['Date'].dt.weekday < 5  # True for Monday-Friday
df['IsWeekend'] = ~df['IsWeekday']          # True for Saturday-Sunday


2. Number of days before and after holiday 

In [47]:
def to_holiday(df, holiday_list):
    # Ensure holiday_list is in datetime format
    holiday_list = pd.to_datetime(holiday_list)
    result = []
    
    for date in df['Date']:
        # Calculate the timedelta to each holiday
        future_holidays = holiday_list[holiday_list > date]
        
        # Find the minimum timedelta in days
        if len(future_holidays) > 0:
            min_days = (future_holidays - date).days.min()
        else:
            min_days = 300  # Arbitrary large value if no future holiday
        
        result.append(min_days)
    return np.array(result)

def after_holiday(df, holiday_list):
    # Ensure holiday_list is in datetime format
    holiday_list = pd.to_datetime(holiday_list)
    result = []
    
    for date in df['Date']:
        # Calculate the timedelta to each holiday
        future_holidays = holiday_list[holiday_list < date]
        
        # Find the minimum timedelta in days
        if len(future_holidays) > 0:
            min_days = (date - future_holidays).days.min()
        else:
            min_days = 300  # Arbitrary large value if no future holiday
        
        result.append(min_days)
    return np.array(result)


In [48]:
from_holiday_a = to_holiday(df, holiday_a)
from_holiday_b = to_holiday(df, holiday_b)
from_holiday_c = to_holiday(df, holiday_c)

after_holiday_a = after_holiday(df, holiday_a)
after_holiday_b = after_holiday(df, holiday_b)
after_holiday_c = after_holiday(df, holiday_c)

In [49]:
df['Days from Holiday_a'] = from_holiday_a
df['Days from Holiday_b'] = from_holiday_b
df['Days from Holiday_c'] = from_holiday_c

df['Days after Holiday_a'] = after_holiday_a
df['Days after Holiday_b'] = after_holiday_b
df['Days after Holiday_c'] = after_holiday_c

In [50]:
import os

data_path = os.path.join("data", 'features')
os.makedirs(data_path)
df.to_csv(os.path.join(data_path, 'added_features.csv'), index=False)
# df.to_csv(os.path.join(data_path, 'test_processed.csv'), index=False)